# 05 · Normalize masters and sales orders

Run this notebook from the `silver/` folder with the existing project kernel. It selects and verifies Bronze manifests, then rebuilds the required prior stages in memory. Reuse the same `silver_settings.json` to review a consistent selection across notebooks.

In [ ]:
from IPython.display import display
import pandas as pd

from silver_notebook import build, frame, input_frame, settings_from_file

model = build("orders", settings_from_file())
display(input_frame(model))

## Declare the grain and inspect the keys

| Entity | One row represents |
|---|---|
| Company | An operating legal entity |
| Source Customer and Source Product | A native account or sellable item in its source scope |
| Warehouse and GL Account | A source location or account inside a company |
| Sales Order | A source order header |
| Sales Order Line | A distinct native item row on a source order |

ERPNext and the dealer use different order representations. The header points to a customer. The line points to a product and retains ordered quantity, UOM and recorded amount. The parent fixture alias P-S1 is for lookup. The actual ERPNext document ID is used in the key. Dealer amounts are converted from INR paise using decimal arithmetic. A missing unit price stays empty.

In [ ]:
display(pd.DataFrame([
    {"table": table, "rows": len(rows)} for table, rows in model.tables.items()
]))
display(frame(model, "source_customer"), frame(model, "source_product"))

In [ ]:
orders = frame(model, "sales_order")
lines = frame(model, "sales_order_line")
if not orders.empty and not lines.empty:
    selected = orders.loc[orders["order_reference"].isin(["P-S1", "D-S1"])]
    display(selected)
    display(lines.loc[lines["sales_order_id"].isin(selected["id"])])
else:
    display(orders, lines)
display(pd.DataFrame(model.exceptions))